# SecureBERT 2.0 + ASL & Bi-Encoder — Kaggle Training Pipeline

Notebook duy nhất để chạy toàn bộ nghiên cứu trên **Kaggle GPU**:

1. SecureBERT 2.0 + ASL — seed 42 và 123.
2. Bi-Encoder Dense Retrieval — seed 42 và 123.
3. Validation-only threshold tuning, test evaluation, raw predictions.
4. Tổng hợp hai seed, tables, error analysis và figures PNG/PDF từ kết quả đã lưu.

**Cách dùng:** tạo Kaggle Notebook, bật GPU, Add Data chứa thư mục `dataset/processed`, upload notebook này và chọn **Run All**. Kết quả nằm tại `/kaggle/working/results`.

In [ ]:
# Kaggle bootstrap. P100 (Pascal/sm_60) needs a PyTorch wheel that still contains its CUDA kernels.
import subprocess, sys
gpu_name=subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],text=True).strip()
print("Detected GPU:",gpu_name)
if "P100" in gpu_name:
    print("[SETUP] Replacing only torch with the P100-compatible CUDA 12.1 wheel (no dependency upgrades)...")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps","torch==2.5.1","--index-url","https://download.pytorch.org/whl/cu121"])
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers>=4.48,<5","iterative-stratification","sentencepiece","tqdm"])
print("[OK] Kaggle dependencies installed")

In [ ]:
from pathlib import Path
import os, re, gc, json, math, time, random, pickle, platform, subprocess, warnings
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi":120,"axes.grid":True,"grid.alpha":0.25,"font.size":10})

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()
RESULTS = WORK_ROOT / "results"
for d in [RESULTS, RESULTS/"tables", RESULTS/"figures", RESULTS/"figures"/"training",
          RESULTS/"figure_data", RESULTS/"aggregated", RESULTS/"error_analysis"]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seeds": [42],
    "securebert_checkpoint": "cisco-ai/SecureBERT2.0-base",
    "biencoder_checkpoint": "cisco-ai/SecureBERT2.0-biencoder",
    "max_length": 384,
    "max_query_length": 384,
    "max_label_length": 64,
    "validation_fraction": 0.10,
    "securebert_epochs": 6,
    "biencoder_epochs": 4,
    "train_batch_size": 8,
    "eval_batch_size": 16,
    "biencoder_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "biencoder_learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "max_grad_norm": 1.0,
    "gamma_neg": 4.0,
    "gamma_pos": 1.0,
    "asl_clip": 0.05,
    "asl_eps": 1e-8,
    "temperature": 0.07,
    "threshold_min": 0.05,
    "threshold_max": 0.95,
    "threshold_step": 0.01,
    "min_val_support_per_label": 5,
    "top_n_cooccurrence": 20,
    "num_workers": 2,
    "log_every_batches": 250,
    "force_rerun": False,
}
RUN_ALL = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Kaggle GPU is not enabled. Open Notebook settings -> Accelerator -> GPU, then Run All.")
gpu_capability=torch.cuda.get_device_capability(0); compiled_arches=torch.cuda.get_arch_list()
required_arch=f"sm_{gpu_capability[0]}{gpu_capability[1]}"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU capability {gpu_capability} | compiled arches {compiled_arches}")
if compiled_arches and required_arch not in compiled_arches:
    raise RuntimeError(f"Installed PyTorch lacks {required_arch}. Restart the Kaggle session and Run All so the compatibility setup cell runs before torch is imported.")
_cuda_smoke=torch.arange(8,device=DEVICE); torch.cuda.synchronize(); del _cuda_smoke
print("[OK] CUDA kernel smoke test passed")
print("Device:", DEVICE, torch.cuda.get_device_name(0))
print("Results:", RESULTS)
print(json.dumps(CONFIG, indent=2))

## 1. Dataset discovery, integrity checks and validation split

The official test set is never used for checkpoint selection or threshold tuning. A validation subset is derived only from the official training set with iterative multi-label stratification, independently for each seed.

In [ ]:
def find_one(filename, required=True):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches and not IS_KAGGLE:
        matches = sorted(Path.cwd().rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(f"Cannot find {filename}. Add the processed dataset as Kaggle input.")
        return None
    preferred = [p for p in matches if "processed" in str(p).lower()]
    chosen = preferred[0] if preferred else matches[0]
    print(f"[DATA] {filename}: {chosen}")
    return chosen


ORIGINAL_TRAIN_PATH = find_one("train_original_fixed.csv")
AUGMENTED_TRAIN_PATH = find_one("train_augmented_eda.csv")
VALIDATION_PATH = find_one("validation_original_fixed.csv")
TEST_PATH = find_one("test.csv")

print("="*57)
print("PROJECT INSPECTION")
print("ORIGINAL TRAIN:", ORIGINAL_TRAIN_PATH)
print("AUGMENTED TRAIN:", AUGMENTED_TRAIN_PATH)
print("VALIDATION:", VALIDATION_PATH)
print("TEST:", TEST_PATH)

original_train_df = pd.read_csv(ORIGINAL_TRAIN_PATH)
train_df = pd.read_csv(AUGMENTED_TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

MLB_PATH = find_one("multilabel_binarizer.pkl")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open(MLB_PATH, "rb") as f: saved_mlb=pickle.load(f)
classes = np.asarray(saved_mlb.classes_, dtype=str)
mlb=MultiLabelBinarizer(classes=classes); mlb.fit([[]])
NUM_LABELS = len(classes)
label_to_idx = {x:i for i,x in enumerate(classes)}

def parse_labels(value):
    return [x.strip() for x in str(value).split(",") if x.strip() and x.strip() != "nan"]
def encode_labels(series):
    return mlb.transform(series.map(parse_labels)).astype(np.float32)

for frame, name in [(original_train_df,"orig_train"), (train_df,"train"), (val_df,"val"), (test_df,"test")]:
    frame["Cleaned_Text"] = frame["Cleaned_Text"].fillna("").astype(str)

y_original_train = encode_labels(original_train_df["Labels"])
y_train = encode_labels(train_df["Labels"])
y_val = encode_labels(val_df["Labels"])
y_test = encode_labels(test_df["Labels"])

original_train_support = y_original_train.sum(axis=0)
augmented_train_support = y_train.sum(axis=0)

# Sanity Checks
print("="*57)
print("RUNNING SANITY CHECKS...")
train_source_ids = set(train_df["source_sample_id"])
val_source_ids = set(val_df.get("source_sample_id", []))
orig_source_ids = set(original_train_df["source_sample_id"])

if train_source_ids & val_source_ids:
    raise RuntimeError("Sanity Check Failed: Overlap between Augmented Train and Validation source_sample_id")

if "is_augmented" in val_df.columns and val_df["is_augmented"].sum() > 0:
    raise RuntimeError("Sanity Check Failed: Validation contains augmented rows")

if "is_augmented" in test_df.columns and test_df["is_augmented"].sum() > 0:
    raise RuntimeError("Sanity Check Failed: Test contains augmented rows")

if "is_augmented" in train_df.columns:
    synth_ids = set(train_df[train_df["is_augmented"] == 1]["source_sample_id"])
    if not synth_ids.issubset(orig_source_ids):
        raise RuntimeError("Sanity Check Failed: Synthetic source_sample_id not in original_train_df")
    
    # Label Preservation Check
    synth_df = train_df[train_df["is_augmented"] == 1]
    parent_map = dict(zip(original_train_df["source_sample_id"], original_train_df["Labels"]))
    for _, row in synth_df.iterrows():
        sid = row["source_sample_id"]
        if sid in parent_map and row["Labels"] != parent_map[sid]:
            raise RuntimeError(f"Sanity Check Failed: Synthetic Labels for {sid} do not match parent Labels")

if len(test_df) != 4453:
    print(f"[WARNING] Expected Test size 4453, but found {len(test_df)}")

print("[OK] Sanity checks passed. No leakage detected.")
print("="*57)

# DEPRECATED:
# Fixed validation is loaded from validation_original_fixed.csv
# validation split function is removed from execution path.


In [ ]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

class TextDataset(Dataset):
    def __init__(self, texts, labels, ids):
        self.texts = list(texts); self.labels = np.asarray(labels, np.float32); self.ids = np.asarray(ids)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.labels[i], self.ids[i]

def make_loader(texts, labels, ids, tokenizer, max_length, batch_size, shuffle, seed):
    ds = TextDataset(texts, labels, ids)
    def collate(batch):
        text, y, sample_id = zip(*batch)
        tok = tokenizer(list(text), padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok["labels"] = torch.tensor(np.stack(y), dtype=torch.float32)
        tok["sample_ids"] = np.asarray(sample_id)
        return tok
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate,
                      num_workers=CONFIG["num_workers"], pin_memory=True,
                      worker_init_fn=seed_worker, generator=g, persistent_workers=CONFIG["num_workers"]>0)

## 2. Central metrics and validation-only threshold tuning

In [ ]:
METRIC_KEYS = ["micro_precision","micro_recall","micro_f1","macro_precision","macro_recall",
               "macro_f1","weighted_f1","hamming_loss"]

def classification_metrics(y_true, y_pred):
    return {
        "micro_precision": precision_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_recall": recall_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_f1": f1_score(y_true,y_pred,average="micro",zero_division=0),
        "macro_precision": precision_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_recall": recall_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_f1": f1_score(y_true,y_pred,average="macro",zero_division=0),
        "weighted_f1": f1_score(y_true,y_pred,average="weighted",zero_division=0),
        "hamming_loss": hamming_loss(y_true,y_pred),
        "avg_predicted_labels": float(y_pred.sum(1).mean()),
    }

def ranking_metrics(y_true, scores, ks=(1,3,5,10,20,50)):
    order = np.argsort(-scores, axis=1)
    true_count = np.maximum(y_true.sum(1), 1)
    out = {}
    for k in ks:
        kk=min(k,scores.shape[1]); hits=np.take_along_axis(y_true,order[:,:kk],axis=1).sum(1)
        out[f"precision_at_{k}"] = float(np.mean(hits/kk))
        out[f"recall_at_{k}"] = float(np.mean(hits/true_count))
        out[f"hit_at_{k}"] = float(np.mean(hits>0))
    ranks=[]; aps=[]
    for i in range(len(y_true)):
        rel=y_true[i,order[i]].astype(bool); pos=np.flatnonzero(rel)
        ranks.append((pos[0]+1) if len(pos) else scores.shape[1]+1)
        if len(pos): aps.append(np.mean([(j+1)/(p+1) for j,p in enumerate(pos)]))
        else: aps.append(0.0)
    out["mrr"]=float(np.mean(1/np.asarray(ranks))); out["map"]=float(np.mean(aps))
    return out

def threshold_sweep(y_true, probs):
    thresholds=np.round(np.arange(CONFIG["threshold_min"], CONFIG["threshold_max"]+1e-9,
                                  CONFIG["threshold_step"]), 10)
    rows=[]
    for t in thresholds:
        m=classification_metrics(y_true,(probs>=t).astype(np.uint8)); m["threshold"]=float(t); rows.append(m)
    return pd.DataFrame(rows)

def tune_thresholds(y_true, probs):
    sweep=threshold_sweep(y_true,probs)
    best_micro=float(sweep.loc[sweep.micro_f1.idxmax(),"threshold"])
    best_macro=float(sweep.loc[sweep.macro_f1.idxmax(),"threshold"])
    support=y_true.sum(0).astype(int); per=np.full(y_true.shape[1],best_micro,dtype=np.float32); rows=[]
    for j in range(y_true.shape[1]):
        fallback=support[j] < CONFIG["min_val_support_per_label"]
        if not fallback:
            vals=[]
            for t in sweep.threshold:
                pred=(probs[:,j]>=t).astype(np.uint8)
                vals.append(f1_score(y_true[:,j],pred,zero_division=0))
            per[j]=float(sweep.threshold.iloc[int(np.argmax(vals))])
        pred=(probs[:,j]>=per[j]).astype(np.uint8)
        rows.append({"technique_id":classes[j],"validation_support":support[j],"optimal_threshold":float(per[j]),
                     "validation_precision":precision_score(y_true[:,j],pred,zero_division=0),
                     "validation_recall":recall_score(y_true[:,j],pred,zero_division=0),
                     "validation_f1":f1_score(y_true[:,j],pred,zero_division=0),"fallback_used":bool(fallback)})
    return best_micro,best_macro,per,sweep,pd.DataFrame(rows)

def label_metrics(y_true,y_pred,train_support,val_support,thresholds,groups):
    rows=[]
    for j,tid in enumerate(classes):
        rows.append({"Technique_ID":tid,"Train_Support":int(train_support[j]),"Validation_Support":int(val_support[j]),
                     "Test_Support":int(y_true[:,j].sum()),"Precision":precision_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Recall":recall_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "F1":f1_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Optimal_Threshold":float(thresholds[j]),"Frequency_Group":groups[j]})
    return pd.DataFrame(rows)

def frequency_groups(original_train_support):
    groups = []
    for x in original_train_support:
        if x >= 100: groups.append("Head")
        elif 30 <= x < 100: groups.append("Medium")
        else: groups.append("Tail")
    return np.asarray(groups), 30.0, 100.0

def print_experiment_summary(result):
    m=result["test_per_label"]
    names=[("Micro Precision","micro_precision"),("Micro Recall","micro_recall"),("Micro-F1","micro_f1"),
           ("Macro Precision","macro_precision"),("Macro Recall","macro_recall"),("Macro-F1","macro_f1"),
           ("Weighted-F1","weighted_f1"),("Hamming Loss","hamming_loss"),
           ("Precision@3","precision_at_3"),("Recall@3","recall_at_3"),("Hit@3","hit_at_3"),
           ("Precision@5","precision_at_5"),("Recall@5","recall_at_5"),("Hit@5","hit_at_5")]
    rows=[{"Metric":label,"Test value":m[key]} for label,key in names if key in m]
    print("\n"+"="*70)
    print(f"EXPERIMENT COMPLETE: {result['model']} | Seed {result['seed']}")
    print("="*70)
    print(f"Best epoch: {result['best_epoch']} | Selection: {result['selection_metric']} = {result['validation_score']:.4f}")
    print(f"Validation-selected global threshold: {result['global_threshold']:.2f}")
    display(pd.DataFrame(rows).set_index("Metric").round(4))
    if result.get("retrieval"):
        retrieval=pd.DataFrame([{"Metric":k.replace("_at_","@").replace("_"," ").title(),"Test value":v} for k,v in result["retrieval"].items()])
        print("Bi-Encoder ranking metrics:"); display(retrieval.set_index("Metric").round(4))
    print(f"Training: {result['training_seconds']/60:.2f} min | Inference: {result['inference_ms_per_sample']:.3f} ms/sample")
    print(f"Peak VRAM: {result['peak_vram_mb']:.1f} MB | Model size: {result['model_size_mb']:.1f} MB | Device: {result['device']}")
    print("="*70)

## 3. Models and losses

ASL consumes raw logits and multi-hot targets. Bi-Encoder uses a multi-positive full-label softmax objective: all ground-truth techniques contribute to the numerator, so a second true label is never treated as a negative. The label encoder is frozen and technique embeddings are precomputed, making training feasible on a Kaggle GPU.

In [ ]:
def masked_mean(last_hidden, attention_mask):
    mask=attention_mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden*mask).sum(1)/mask.sum(1).clamp_min(1e-9)

def load_securebert_encoder(checkpoint):
    model_config=AutoConfig.from_pretrained(checkpoint)
    if hasattr(model_config,"reference_compile"): model_config.reference_compile=False
    return AutoModel.from_pretrained(checkpoint,config=model_config,attn_implementation="eager")

class SecureClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels):
        super().__init__(); self.encoder=load_securebert_encoder(checkpoint)
        hidden=self.encoder.config.hidden_size
        self.dropout=nn.Dropout(getattr(self.encoder.config,"classifier_dropout",0.1) or 0.1)
        self.classifier=nn.Linear(hidden,num_labels)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        return self.classifier(self.dropout(masked_mean(out.last_hidden_state,attention_mask)))

class AsymmetricLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,logits,targets):
        p=torch.sigmoid(logits); pos=p; neg=1-p
        if CONFIG["asl_clip"]:
            neg=(neg+CONFIG["asl_clip"]).clamp(max=1)
        loss=targets*torch.log(pos.clamp_min(CONFIG["asl_eps"]))+(1-targets)*torch.log(neg.clamp_min(CONFIG["asl_eps"]))
        pt=pos*targets+neg*(1-targets)
        weight=torch.pow((1-pt).clamp_min(0), CONFIG["gamma_pos"]*targets+CONFIG["gamma_neg"]*(1-targets))
        return -(loss*weight).sum(1).mean()

class Encoder(nn.Module):
    def __init__(self,checkpoint):
        super().__init__(); self.backbone=load_securebert_encoder(checkpoint)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask)
        return F.normalize(masked_mean(out.last_hidden_state,attention_mask),p=2,dim=1)

# ASL sanity checks
_loss=AsymmetricLoss(); _z=torch.tensor([[5.,-5.],[-5.,5.]],requires_grad=True); _y=torch.tensor([[1.,0.],[0.,1.]])
_good=_loss(_z,_y); _bad=_loss(-_z,_y); _good.backward()
assert torch.isfinite(_good) and _good<_bad and _z.grad is not None and torch.isfinite(_z.grad).all()
print("[OK] ASL finite, direction and gradient checks passed")

In [ ]:
def optimizer_and_scheduler(model, loader_len, epochs, lr):
    no_decay=("bias","LayerNorm.weight","layer_norm.weight")
    params=[{"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],"weight_decay":CONFIG["weight_decay"]},
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],"weight_decay":0.0}]
    opt=torch.optim.AdamW(params,lr=lr)
    steps=math.ceil(loader_len/CONFIG["gradient_accumulation_steps"])*epochs
    sch=get_linear_schedule_with_warmup(opt,int(steps*CONFIG["warmup_ratio"]),steps)
    return opt,sch

def to_device(batch):
    ids=batch.pop("sample_ids"); y=batch.pop("labels").to(DEVICE,non_blocking=True)
    x={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
    return x,y,ids

@torch.no_grad()
def predict_classifier(model,loader,desc="Evaluating SecureBERT"):
    model.eval(); ys=[]; probs=[]; ids=[]
    progress=tqdm(loader,desc=desc,leave=False,dynamic_ncols=True)
    for batch in progress:
        x,y,sid=to_device(batch); logits=model(**x)
        ys.append(y.cpu().numpy()); probs.append(torch.sigmoid(logits).cpu().numpy()); ids.extend(sid.tolist())
    return np.concatenate(ys),np.concatenate(probs),np.asarray(ids)

def train_classifier_epoch(model,loader,loss_fn,opt,sch,scaler,desc):
    model.train(); opt.zero_grad(set_to_none=True); total=0
    progress=tqdm(enumerate(loader,1),total=len(loader),desc=desc,leave=True,dynamic_ncols=True)
    for step,batch in progress:
        x,y,_=to_device(batch)
        with torch.autocast("cuda",dtype=torch.float16):
            loss=loss_fn(model(**x),y)/CONFIG["gradient_accumulation_steps"]
        if not torch.isfinite(loss): raise FloatingPointError("NaN/Inf ASL loss")
        scaler.scale(loss).backward(); batch_loss=loss.item()*CONFIG["gradient_accumulation_steps"]; total+=batch_loss
        if step%CONFIG["gradient_accumulation_steps"]==0 or step==len(loader):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG["max_grad_norm"])
            old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale()>=old_scale: sch.step()
            else: print(f"[AMP] Optimizer step skipped after gradient overflow at batch {step}; scheduler unchanged.",flush=True)
            opt.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{batch_loss:.4f}",avg=f"{total/step:.4f}",lr=f"{opt.param_groups[0]['lr']:.2e}",vram=f"{torch.cuda.memory_allocated()/1024**3:.1f}GB")
        if step%CONFIG["log_every_batches"]==0 or step==len(loader): print(f"[PROGRESS] {desc} | batch {step}/{len(loader)} | loss={batch_loss:.4f} | avg={total/step:.4f} | lr={opt.param_groups[0]['lr']:.2e} | VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB",flush=True)
    return total/len(loader)

## 4. SecureBERT 2.0 + ASL training — seed 42 and 123

In [ ]:
def run_securebert(seed):
    run_dir = RESULTS / "securebert_asl_augmented" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir/"metrics.json").exists() and not CONFIG["force_rerun"]:
        print(f"[SKIP] SecureBERT seed {seed} completed"); return
    
    print(f"
{'='*70}
TRAINING SECUREBERT 2.0 + ASL | SEED {seed}
{'='*70}")
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); started=time.time()
    
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["securebert_checkpoint"])
    
    train_loader = make_loader(train_df.Cleaned_Text, y_train, np.arange(len(train_df)), tokenizer, CONFIG["max_length"], CONFIG["train_batch_size"], True, seed)
    val_loader = make_loader(val_df.Cleaned_Text, y_val, np.arange(len(val_df)), tokenizer, CONFIG["max_length"], CONFIG["eval_batch_size"], False, seed)
    test_loader = make_loader(test_df.Cleaned_Text, y_test, np.arange(len(test_df)), tokenizer, CONFIG["max_length"], CONFIG["eval_batch_size"], False, seed)
    
    model = SecureClassifier(CONFIG["securebert_checkpoint"], NUM_LABELS).to(DEVICE)
    opt, sch = optimizer_and_scheduler(model, len(train_loader), CONFIG["securebert_epochs"], CONFIG["learning_rate"])
    scaler = torch.cuda.amp.GradScaler(); loss_fn = AsymmetricLoss(); history = []; best = -1; best_epoch = 0
    checkpoint = run_dir / "best_model.pt"
    
    epoch_times = []
    
    with open(run_dir / "training.log", "w") as f_log:
        f_log.write("=== SecureBERT Training Log ===
")
        
        for epoch in range(1, CONFIG["securebert_epochs"]+1):
            t = time.time()
            loss = train_classifier_epoch(model, train_loader, loss_fn, opt, sch, scaler, desc=f"SecureBERT seed {seed} | epoch {epoch}/{CONFIG['securebert_epochs']}")
            epoch_time = time.time() - t
            epoch_times.append(epoch_time)
            
            vy, vp, _ = predict_classifier(model, val_loader, desc=f"Validation seed {seed} | epoch {epoch}")
            vm = classification_metrics(vy, (vp>=0.5).astype(np.uint8))
            
            history.append({"epoch":epoch, "train_loss":loss, "val_micro_f1":vm["micro_f1"], "val_macro_f1":vm["macro_f1"], "seconds":epoch_time})
            
            log_str = (
                f"
Epoch {epoch}
"
                f"Train Loss: {loss:.6f}
"
                f"Validation Micro Precision: {vm['micro_precision']*100:.2f}%
"
                f"Validation Micro Recall: {vm['micro_recall']*100:.2f}%
"
                f"Validation Micro F1: {vm['micro_f1']*100:.2f}%
"
                f"Validation Macro Precision: {vm['macro_precision']*100:.2f}%
"
                f"Validation Macro Recall: {vm['macro_recall']*100:.2f}%
"
                f"Validation Macro F1: {vm['macro_f1']*100:.2f}%
"
                f"Validation Weighted F1: {vm['weighted_f1']*100:.2f}%
"
                f"Validation Hamming Loss: {vm['hamming_loss']:.6f}
"
                f"Learning Rate: {opt.param_groups[0]['lr']:.4e}
"
                f"Epoch Time: {epoch_time:.2f} sec
"
            )
            print(log_str)
            f_log.write(log_str)
            
            if vm["macro_f1"] > best:
                best = vm["macro_f1"]
                best_epoch = epoch
                torch.save(model.state_dict(), checkpoint)
                
            best_log = f"Best Epoch So Far: {best_epoch}
Best Validation Macro F1 @ 0.5: {best*100:.2f}%
======================================================================"
            print(best_log)
            f_log.write(best_log + "
")
            
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    vy, vp, _ = predict_classifier(model, val_loader, desc=f"Final validation seed {seed}")
    
    global_t, macro_t, per_t, sweep, threshold_table = tune_thresholds(vy, vp)
    sweep.to_csv(run_dir / "threshold_sweep.csv", index=False)
    threshold_table.to_csv(run_dir / "per_label_thresholds.csv", index=False)
    
    infer_start = time.perf_counter()
    ty, tp, tids = predict_classifier(model, test_loader, desc=f"Locked test inference seed {seed}")
    torch.cuda.synchronize()
    infer_seconds = time.perf_counter() - infer_start
    
    pred_global = (tp >= global_t).astype(np.uint8)
    pred_per = (tp >= per_t[None, :]).astype(np.uint8)
    
    global_metrics = classification_metrics(ty, pred_global)
    global_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))
    
    per_metrics = classification_metrics(ty, pred_per)
    per_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))
    
    val_support = vy.sum(0).astype(int)
    groups, q25, q75 = frequency_groups(original_train_support)
    
    per_label = label_metrics(ty, pred_per, augmented_train_support, val_support, per_t, groups)
    per_label["Original_Train_Support"] = original_train_support.astype(int)
    per_label["Augmented_Train_Support"] = per_label.pop("Train_Support")
    per_label.to_csv(run_dir / "per_label_metrics.csv", index=False)
    
    pd.DataFrame(history).to_csv(run_dir / "epoch_history.csv", index=False)
    np.savez_compressed(run_dir / "predictions.npz", sample_ids=tids, y_true=ty, probabilities=tp,
                        global_predictions=pred_global, per_label_predictions=pred_per,
                        global_threshold=np.array(global_t), per_label_thresholds=per_t)
                        
    params = sum(p.numel() for p in model.parameters())
    model_mb = checkpoint.stat().st_size / 1024**2
    
    # Dataset statistics
    dataset_sizes = {
        "original_fixed_train_size": len(original_train_df),
        "augmented_train_size": len(train_df),
        "original_rows_in_augmented": int(len(train_df[train_df.get('is_augmented', 0) == 0])),
        "synthetic_rows": int(len(train_df[train_df.get('is_augmented', 0) == 1])),
        "validation_size": len(val_df),
        "test_size": len(test_df),
        "num_labels": NUM_LABELS,
        "single_label_count": int((y_train.sum(1) == 1).sum()),
        "multi_label_count": int((y_train.sum(1) > 1).sum()),
        "average_labels_per_sample": float(y_train.sum(1).mean()),
        "support_min": int(original_train_support.min()),
        "support_max": int(original_train_support.max()),
        "support_mean": float(original_train_support.mean()),
        "Head_count": int((groups == "Head").sum()),
        "Medium_count": int((groups == "Medium").sum()),
        "Tail_count": int((groups == "Tail").sum()),
    }
    pd.DataFrame([dataset_sizes]).to_csv(run_dir / "dataset_statistics.csv", index=False)
    
    comp_cost = {
        "training_seconds": time.time() - started,
        "training_minutes": (time.time() - started) / 60,
        "inference_ms_per_sample": infer_seconds / len(ty) * 1000,
        "seconds_per_1000_samples": (infer_seconds / len(ty)) * 1000,
        "samples_per_second": len(ty) / infer_seconds,
        "peak_cpu_ram_mb": psutil.Process().memory_info().rss / 1024**2,
        "peak_vram_allocated_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "peak_vram_reserved_mb": torch.cuda.max_memory_reserved() / 1024**2,
        "model_size_mb": model_mb,
        "total_parameters": params,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "device": torch.cuda.get_device_name(0)
    }
    pd.DataFrame([comp_cost]).to_csv(run_dir / "computational_cost.csv", index=False)
    
    # Frequency group metrics
    fg_rows = []
    for gname in ["Head", "Medium", "Tail"]:
        g_df = per_label[per_label["Frequency_Group"] == gname]
        g_df_active = g_df[g_df["Test_Support"] > 0]
        fg_rows.append({
            "Frequency_Group": gname,
            "Label_Count": len(g_df),
            "Mean_P": g_df["Precision"].mean(),
            "Median_P": g_df["Precision"].median(),
            "Mean_R": g_df["Recall"].mean(),
            "Median_R": g_df["Recall"].median(),
            "Mean_F1": g_df["F1"].mean(),
            "Median_F1": g_df["F1"].median(),
            "F1=0_Count": int((g_df["F1"] == 0).sum()),
            "F1=0_%": float((g_df["F1"] == 0).mean() * 100),
            "F1=1_Count": int((g_df["F1"] == 1).sum()),
            "F1=1_%": float((g_df["F1"] == 1).mean() * 100),
            "Total_Test_Support": int(g_df["Test_Support"].sum())
        })
    fg_df = pd.DataFrame(fg_rows)
    fg_df.to_csv(run_dir / "frequency_group_performance.csv", index=False)
    
    tail_present = per_label[(per_label["Frequency_Group"] == "Tail") & (per_label["Test_Support"] > 0)]
    
    result = {
        "model": "SecureBERT 2.0 + ASL",
        "loss": "Asymmetric Loss",
        "seed": seed,
        "augmentation": True,
        "augmentation_mode": "eda",
        "protocol": "leakage_safe_fixed_validation",
        "best_epoch": best_epoch,
        "global_threshold": float(global_t),
        "test_global": global_metrics,
        "test_per_label": per_metrics,
        "frequency_group_metrics": fg_df.to_dict("records"),
        "tail_present_metrics": {
            "count": len(tail_present),
            "mean_f1": float(tail_present["F1"].mean()) if len(tail_present) else 0,
            "median_f1": float(tail_present["F1"].median()) if len(tail_present) else 0,
            "f1=0_count": int((tail_present["F1"] == 0).sum()),
            "f1=0_%": float((tail_present["F1"] == 0).mean() * 100) if len(tail_present) else 0,
        },
        "computational_metrics": comp_cost,
        "dataset_sizes": dataset_sizes
    }
    (run_dir / "metrics.json").write_text(json.dumps(result, indent=2))
    tokenizer.save_pretrained(run_dir / "tokenizer")
    
    print(f"
Best Epoch: {best_epoch}")
    print(f"Global Threshold: {global_t:.2f}
")
    print("TEST GLOBAL")
    print(f"Micro P: {global_metrics['micro_precision']*100:.2f}%")
    print(f"Micro R: {global_metrics['micro_recall']*100:.2f}%")
    print(f"Micro F1: {global_metrics['micro_f1']*100:.2f}%")
    print(f"Macro P: {global_metrics['macro_precision']*100:.2f}%")
    print(f"Macro R: {global_metrics['macro_recall']*100:.2f}%")
    print(f"Macro F1: {global_metrics['macro_f1']*100:.2f}%")
    print(f"Weighted F1: {global_metrics['weighted_f1']*100:.2f}%")
    print(f"Hamming Loss: {global_metrics['hamming_loss']:.6f}
")
    print(f"P@3: {global_metrics.get('precision_at_3', 0)*100:.2f}%")
    print(f"R@3: {global_metrics.get('recall_at_3', 0)*100:.2f}%")
    print(f"Hit@3: {global_metrics.get('hit_at_3', 0)*100:.2f}%")
    print(f"P@5: {global_metrics.get('precision_at_5', 0)*100:.2f}%")
    print(f"R@5: {global_metrics.get('recall_at_5', 0)*100:.2f}%")
    print(f"Hit@5: {global_metrics.get('hit_at_5', 0)*100:.2f}%")
    print(f"MRR: {global_metrics.get('mrr', 0):.4f}")
    print(f"MAP: {global_metrics.get('map', 0):.4f}
")
    print("---")
    print(f"HEAD
Mean F1: {fg_df[fg_df['Frequency_Group']=='Head']['Mean_F1'].iloc[0]*100:.2f}%
")
    print(f"MEDIUM
Mean F1: {fg_df[fg_df['Frequency_Group']=='Medium']['Mean_F1'].iloc[0]*100:.2f}%
")
    print(f"TAIL
Mean F1: {fg_df[fg_df['Frequency_Group']=='Tail']['Mean_F1'].iloc[0]*100:.2f}%
")
    print(f"Tail labels present in Test: {result['tail_present_metrics']['count']}")
    print(f"Tail-present Mean F1: {result['tail_present_metrics']['mean_f1']*100:.2f}%")
    print(f"Tail-present F1=0: {result['tail_present_metrics']['f1=0_count']} / {result['tail_present_metrics']['count']}
")
    print("---")
    print(f"Training Time: {comp_cost['training_minutes']:.2f} min")
    print(f"Inference: {comp_cost['inference_ms_per_sample']:.4f} ms/sample")
    print(f"Peak VRAM: {comp_cost['peak_vram_allocated_mb']:.0f} MB")
    print(f"Model Size: {comp_cost['model_size_mb']:.2f} MB")
    print("======================================================================")
    
    del model, opt, sch, scaler, train_loader, val_loader, test_loader; gc.collect(); torch.cuda.empty_cache()

## 5. Bi-Encoder Dense Retrieval training — seed 42 and 123

No external ATT&CK metadata file is required. Technique representations use only the 378 Technique IDs from `multilabel_binarizer.pkl`. This is recorded as an **ID-only Bi-Encoder** configuration in every run.

In [ ]:
def build_technique_texts():
    texts=[f"MITRE ATT&CK Technique ID [{tid}]" for tid in classes]
    print(f"[INFO] Bi-Encoder label representation: ID-only ({len(texts)} Technique IDs); no Attack_Dataset.csv used.")
    return texts

technique_texts=build_technique_texts()

@torch.no_grad()
def encode_label_texts(model,tokenizer):
    model.eval(); chunks=[]
    starts=range(0,NUM_LABELS,CONFIG["eval_batch_size"])
    for i in tqdm(starts,total=len(starts),desc="Encoding technique labels",leave=False,dynamic_ncols=True):
        tok=tokenizer(technique_texts[i:i+CONFIG["eval_batch_size"]],padding=True,truncation=True,
                      max_length=CONFIG["max_label_length"],return_tensors="pt").to(DEVICE)
        chunks.append(model(**tok).cpu())
    return torch.cat(chunks).to(DEVICE)

@torch.no_grad()
def predict_biencoder(model,loader,label_embeddings,desc="Evaluating Bi-Encoder"):
    model.eval(); ys=[]; scores=[]; ids=[]
    progress=tqdm(loader,desc=desc,leave=False,dynamic_ncols=True)
    for batch in progress:
        x,y,sid=to_device(batch); q=model(**x); scores.append((q@label_embeddings.T).cpu().numpy())
        ys.append(y.cpu().numpy()); ids.extend(sid.tolist())
    return np.concatenate(ys),np.concatenate(scores),np.asarray(ids)

def train_biencoder_epoch(model,loader,label_embeddings,opt,sch,scaler,desc):
    model.train(); opt.zero_grad(set_to_none=True); total=0
    progress=tqdm(enumerate(loader,1),total=len(loader),desc=desc,leave=True,dynamic_ncols=True)
    for step,batch in progress:
        x,y,_=to_device(batch)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=(model(**x)@label_embeddings.T)/CONFIG["temperature"]
            positive_logits=logits.masked_fill(y<0.5,-torch.inf)
            valid=y.sum(1)>0
            loss=-(torch.logsumexp(positive_logits[valid],1)-torch.logsumexp(logits[valid],1)).mean()
            loss=loss/CONFIG["gradient_accumulation_steps"]
        if not torch.isfinite(loss): raise FloatingPointError("NaN/Inf bi-encoder loss")
        scaler.scale(loss).backward(); batch_loss=loss.item()*CONFIG["gradient_accumulation_steps"]; total+=batch_loss
        if step%CONFIG["gradient_accumulation_steps"]==0 or step==len(loader):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG["max_grad_norm"])
            old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale()>=old_scale: sch.step()
            else: print(f"[AMP] Optimizer step skipped after gradient overflow at batch {step}; scheduler unchanged.",flush=True)
            opt.zero_grad(set_to_none=True)
        progress.set_postfix(loss=f"{batch_loss:.4f}",avg=f"{total/step:.4f}",lr=f"{opt.param_groups[0]['lr']:.2e}",vram=f"{torch.cuda.memory_allocated()/1024**3:.1f}GB")
        if step%CONFIG["log_every_batches"]==0 or step==len(loader): print(f"[PROGRESS] {desc} | batch {step}/{len(loader)} | loss={batch_loss:.4f} | avg={total/step:.4f} | lr={opt.param_groups[0]['lr']:.2e} | VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB",flush=True)
    return total/len(loader)

def run_biencoder(seed):
    run_dir=RESULTS/"biencoder"/f"seed_{seed}"; run_dir.mkdir(parents=True,exist_ok=True)
    if (run_dir/"metrics.json").exists() and not CONFIG["force_rerun"]:
        print(f"[SKIP] Bi-Encoder seed {seed} completed"); return
    print(f"\n{'='*70}\nTRAINING BI-ENCODER DENSE RETRIEVAL | SEED {seed}\n{'='*70}")
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); started=time.time()
    tokenizer=AutoTokenizer.from_pretrained(CONFIG["biencoder_checkpoint"])
    train_loader=make_loader(train_full_df.Cleaned_Text.iloc[tr_idx],y_train_full[tr_idx],tr_idx,tokenizer,CONFIG["max_query_length"],CONFIG["biencoder_batch_size"],True,seed)
    val_loader=make_loader(train_full_df.Cleaned_Text.iloc[va_idx],y_train_full[va_idx],va_idx,tokenizer,CONFIG["max_query_length"],CONFIG["eval_batch_size"],False,seed)
    test_loader=make_loader(test_df.Cleaned_Text,y_test,np.arange(len(test_df)),tokenizer,CONFIG["max_query_length"],CONFIG["eval_batch_size"],False,seed)
    query_encoder=Encoder(CONFIG["biencoder_checkpoint"]).to(DEVICE)
    label_encoder=Encoder(CONFIG["biencoder_checkpoint"]).to(DEVICE)
    for p in label_encoder.parameters(): p.requires_grad=False
    label_embeddings=encode_label_texts(label_encoder,tokenizer)
    label_params=sum(p.numel() for p in label_encoder.parameters())
    np.save(run_dir/"technique_embeddings.npy",label_embeddings.cpu().numpy())
    del label_encoder; gc.collect(); torch.cuda.empty_cache()
    opt,sch=optimizer_and_scheduler(query_encoder,len(train_loader),CONFIG["biencoder_epochs"],CONFIG["biencoder_learning_rate"])
    scaler=torch.cuda.amp.GradScaler(); history=[]; best=-1; best_epoch=0; checkpoint=run_dir/"best_query_encoder.pt"
    for epoch in range(1,CONFIG["biencoder_epochs"]+1):
        t=time.time(); loss=train_biencoder_epoch(query_encoder,train_loader,label_embeddings,opt,sch,scaler,desc=f"Bi-Encoder seed {seed} | epoch {epoch}/{CONFIG['biencoder_epochs']}")
        vy,vs,_=predict_biencoder(query_encoder,val_loader,label_embeddings,desc=f"Validation seed {seed} | epoch {epoch}"); vm=ranking_metrics(vy,vs,ks=(3,5,10))
        history.append({"epoch":epoch,"train_loss":loss,"val_recall_at_5":vm["recall_at_5"],"val_hit_at_5":vm["hit_at_5"],"seconds":time.time()-t})
        print(f"BiEncoder seed={seed} epoch={epoch}: loss={loss:.4f} val R@5={vm['recall_at_5']:.4f}")
        if vm["recall_at_5"]>best:
            best=vm["recall_at_5"]; best_epoch=epoch; torch.save(query_encoder.state_dict(),checkpoint)
    query_encoder.load_state_dict(torch.load(checkpoint,map_location=DEVICE))
    vy,vs,_=predict_biencoder(query_encoder,val_loader,label_embeddings,desc=f"Final validation seed {seed}")
    # Cosine similarities are converted deterministically to [0,1]; thresholds are tuned only on validation.
    vp=1/(1+np.exp(-vs/CONFIG["temperature"]))
    global_t,macro_t,per_t,sweep,threshold_table=tune_thresholds(vy,vp)
    sweep.to_csv(run_dir/"threshold_sweep.csv",index=False); threshold_table.to_csv(run_dir/"per_label_thresholds.csv",index=False)
    infer_start=time.perf_counter(); ty,ts,tids=predict_biencoder(query_encoder,test_loader,label_embeddings,desc=f"Locked test retrieval seed {seed}"); torch.cuda.synchronize(); infer_seconds=time.perf_counter()-infer_start
    tp=1/(1+np.exp(-ts/CONFIG["temperature"])); pred_global=(tp>=global_t).astype(np.uint8); pred_per=(tp>=per_t[None,:]).astype(np.uint8)
    retrieval=ranking_metrics(ty,ts,ks=(1,3,5,10,20,50)); global_metrics=classification_metrics(ty,pred_global); per_metrics=classification_metrics(ty,pred_per)
    per_metrics.update({k:v for k,v in retrieval.items() if k in {"precision_at_3","recall_at_3","hit_at_3","precision_at_5","recall_at_5","hit_at_5"}})
    train_support=y_train_full.sum(0).astype(int); val_support=vy.sum(0).astype(int); groups,q25,q75=frequency_groups(train_support)
    per_label=label_metrics(ty,pred_per,train_support,val_support,per_t,groups); per_label.to_csv(run_dir/"per_label_metrics.csv",index=False)
    order=np.argsort(-ts,axis=1); inverse=np.argsort(order,axis=1)+1
    pd.DataFrame(history).to_csv(run_dir/"training_history.csv",index=False)
    np.savez_compressed(run_dir/"retrieval_predictions.npz",sample_ids=tids,y_true=ty,similarities=ts,
                        rankings=order,true_label_ranks=inverse,probabilities=tp,
                        global_predictions=pred_global,per_label_predictions=pred_per)
    params=sum(p.numel() for p in query_encoder.parameters())+label_params
    model_mb=(checkpoint.stat().st_size+(run_dir/"technique_embeddings.npy").stat().st_size)/1024**2
    result={"model":"Bi-Encoder Dense Retrieval","seed":seed,"best_epoch":best_epoch,
            "label_representation":"Technique ID only; no name or description metadata",
            "selection_metric":"validation Recall@5","validation_score":best,"global_threshold":global_t,
            "macro_optimal_validation_threshold":macro_t,"retrieval":retrieval,"test_global":global_metrics,
            "test_per_label":per_metrics,"training_seconds":time.time()-started,
            "inference_ms_per_sample":infer_seconds/len(ty)*1000,"peak_vram_mb":torch.cuda.max_memory_allocated()/1024**2,
            "model_size_mb":model_mb,"parameters":params,"trainable_parameters":sum(p.numel() for p in query_encoder.parameters()),
            "device":torch.cuda.get_device_name(0),"train_size":len(tr_idx),"val_size":len(va_idx),"test_size":len(test_df),
            "frequency_q25":q25,"frequency_q75":q75}
    (run_dir/"metrics.json").write_text(json.dumps(result,indent=2)); (run_dir/"config.json").write_text(json.dumps({**CONFIG,"seed":seed,"label_representation":result["label_representation"]},indent=2))
    (run_dir/"model_description.txt").write_text("Bi-Encoder Dense Retrieval with a trainable SecureBERT 2.0 query encoder and frozen SecureBERT 2.0 label encoder. Label input contains Technique ID only; no Attack_Dataset.csv, technique name, or description is used. Training uses a multi-positive full-label contrastive objective.")
    print_experiment_summary(result)
    print(f"[OK] Bi-Encoder seed {seed} -> {run_dir}")
    del query_encoder,opt,sch,scaler,train_loader,val_loader,test_loader,label_embeddings; gc.collect(); torch.cuda.empty_cache()

## 6. Aggregate actual outputs and generate paper figures

This section only reads saved JSON/CSV/NPZ files. It never starts training and never replaces missing values with zero.

In [ ]:
def savefig(fig,name):
    fig.tight_layout(); fig.savefig(RESULTS/"figures"/f"{name}.png",dpi=300,bbox_inches="tight")
    fig.savefig(RESULTS/"figures"/f"{name}.pdf",bbox_inches="tight"); plt.close(fig)

def read_metric(model_dir,seed):
    p=RESULTS/model_dir/f"seed_{seed}"/"metrics.json"
    return json.loads(p.read_text()) if p.exists() else None

def aggregate_and_tables():
    rows=[]
    for model_dir in ["securebert_asl_augmented"]:
        for seed in CONFIG["seeds"]:
            m=read_metric(model_dir,seed)
            if m:
                row={"Model":m["model"],"Seed":seed,**m["test_per_label"],**{f"retrieval_{k}":v for k,v in m.get("retrieval",{}).items()},
                     "training_seconds":m["training_seconds"],"inference_ms_per_sample":m["inference_ms_per_sample"],
                     "model_size_mb":m["model_size_mb"],"parameters":m["parameters"],"device":m["device"]}; rows.append(row)
    raw=pd.DataFrame(rows); raw.to_csv(RESULTS/"aggregated"/"all_runs.csv",index=False)
    if raw.empty: return raw
    comparison_cols=["micro_precision","micro_recall","micro_f1","macro_precision","macro_recall","macro_f1","weighted_f1","hamming_loss","precision_at_3","recall_at_3","hit_at_3","precision_at_5","recall_at_5","hit_at_5"]
    table2=raw.groupby("Model")[comparison_cols].mean().reset_index()
    table2.to_csv(RESULTS/"tables"/"table2_model_comparison.csv",index=False)
    print("\nTABLE 2 — MODEL COMPARISON (TWO-SEED MEAN)"); display(table2.round(4))
    table4=raw.groupby("Model").agg(Training_time_seconds=("training_seconds","mean"),Inference_ms_per_sample=("inference_ms_per_sample","mean"),Model_size_MB=("model_size_mb","mean"),Parameters=("parameters","mean"),Device=("device","first")).reset_index()
    table4.to_csv(RESULTS/"tables"/"table4_computational_cost.csv",index=False)
    print("\nTABLE 4 — COMPUTATIONAL COST"); display(table4.round(4))
    metrics=["micro_f1","macro_f1","weighted_f1","hamming_loss","precision_at_3","recall_at_3","hit_at_3","precision_at_5","recall_at_5","hit_at_5"]
    stability=[]
    for model,g in raw.groupby("Model"):
        for metric in metrics:
            vals={int(r.Seed):r[metric] for _,r in g.iterrows()}
            if all(s in vals for s in CONFIG["seeds"]):
                a,b=vals[42],vals[123]
                stability.append({"Model":model,"Metric":metric,"Seed_42":a,"Seed_123":b,"Mean":np.mean([a,b]),"Standard_Deviation":np.std([a,b],ddof=1),"Absolute_Difference":abs(a-b)})
    stability_df=pd.DataFrame(stability); stability_df.to_csv(RESULTS/"tables"/"table5_two_seed_reproducibility.csv",index=False)
    print("\nTABLE 5 — REPRODUCIBILITY ACROSS TWO RANDOM SEEDS"); display(stability_df.round(4))
    return raw

def generate_descriptive_figures():
    train_support=y_train_full.sum(0).astype(int); groups,_,_=frequency_groups(train_support)
    d=pd.DataFrame({"Technique_ID":classes,"Train_Support":train_support,"Frequency_Group":groups}).sort_values("Train_Support",ascending=False)
    d.to_csv(RESULTS/"figure_data"/"fig1_label_distribution.csv",index=False)
    fig,ax=plt.subplots(figsize=(10,max(12,NUM_LABELS*.16))); ax.barh(d.Technique_ID[::-1],d.Train_Support[::-1],color="#376795"); ax.set(xlabel="Number of training samples",ylabel="Technique ID",title="Training label distribution")
    ax.tick_params(axis="y",labelsize=5); savefig(fig,"fig1_label_distribution")
    counts=pd.concat([train_full_df.Labels,test_df.Labels]).map(lambda x:len(parse_labels(x)))
    cats=pd.Categorical(counts.map(lambda x:str(x) if x<4 else ">=4"),categories=["1","2","3",">=4"],ordered=True)
    d2=pd.Series(cats).value_counts(sort=False).rename_axis("Label_Count_Group").reset_index(name="Sample_Count"); d2["Percentage"]=100*d2.Sample_Count/d2.Sample_Count.sum(); d2.to_csv(RESULTS/"figure_data"/"fig2_labels_per_sample.csv",index=False)
    fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(d2.Label_Count_Group,d2.Sample_Count,color="#4C956C")
    for b,(_,r) in zip(bars,d2.iterrows()): ax.text(b.get_x()+b.get_width()/2,b.get_height(),f"{int(r.Sample_Count):,}\n({r.Percentage:.1f}%)",ha="center",va="bottom")
    ax.set(xlabel="Labels per sample",ylabel="Sample count",title="Number of labels per CTI sample"); savefig(fig,"fig2_labels_per_sample")
    all_y=np.vstack([y_train_full,y_test]); top=np.argsort(-train_support)[:CONFIG["top_n_cooccurrence"]]; co=all_y[:,top].T@all_y[:,top]
    co_df=pd.DataFrame(co.astype(int),index=classes[top],columns=classes[top]); co_df.to_csv(RESULTS/"figure_data"/"fig3_label_cooccurrence.csv")
    fig,ax=plt.subplots(figsize=(11,9)); image=ax.imshow(co_df.to_numpy(),cmap="Blues",aspect="auto"); ax.grid(False); ax.set_xticks(range(len(co_df)),co_df.columns,rotation=60,ha="right"); ax.set_yticks(range(len(co_df)),co_df.index); fig.colorbar(image,ax=ax,label="Co-occurrence count"); ax.set_title("Label co-occurrence (top 20 training labels)"); savefig(fig,"fig3_label_cooccurrence")
    print("[OK] Figures 1–3")

def generate_result_figures(raw):
    # Figure 4: validation threshold curves, both seeds and mean.
    curves=[]
    for seed in CONFIG["seeds"]:
        p=RESULTS/"securebert_asl_augmented"/f"seed_{seed}"/"threshold_sweep.csv"
        if p.exists(): q=pd.read_csv(p); q["Seed"]=seed; curves.append(q)
    if len(curves)==2:
        allc=pd.concat(curves); mean=allc.groupby("threshold",as_index=False).mean(numeric_only=True); allc.to_csv(RESULTS/"figure_data"/"fig4_threshold_performance.csv",index=False)
        fig,ax=plt.subplots(figsize=(8,5))
        for col,label in [("micro_f1","Micro-F1"),("macro_f1","Macro-F1"),("micro_precision","Micro Precision"),("micro_recall","Micro Recall")]: ax.plot(mean.threshold,mean[col],label=label)
        ax.axvline(mean.loc[mean.micro_f1.idxmax(),"threshold"],ls="--",color="black",label="Best mean Micro-F1"); ax.set(xlabel="Global validation threshold",ylabel="Score",title="Validation threshold-performance relationship"); ax.legend(); savefig(fig,"fig4a_threshold_metrics")
        fig,ax=plt.subplots(figsize=(8,4)); ax.plot(mean.threshold,mean.avg_predicted_labels,color="#9C6644"); ax.set(xlabel="Global validation threshold",ylabel="Average predicted labels/sample",title="Validation prediction cardinality"); savefig(fig,"fig4b_threshold_cardinality")
        print("[OK] Figure 4")
    else: print("[UNAVAILABLE] Figure 4 requires both SecureBERT seeds")
    if not raw.empty and raw.groupby("Model").Seed.nunique().min()>=2:
        metrics=[("micro_f1","Micro-F1"),("macro_f1","Macro-F1"),("weighted_f1","Weighted-F1")]; rows=[]
        for model,g in raw.groupby("Model"):
            for key,label in metrics:
                vals=g.set_index("Seed")[key]; rows.append({"Model":model,"Metric":label,"Seed_42":vals.get(42,np.nan),"Seed_123":vals.get(123,np.nan),"Mean":vals.mean(),"Std":vals.std(ddof=1)})
        d=pd.DataFrame(rows); d.to_csv(RESULTS/"figure_data"/"fig5_model_comparison.csv",index=False)
        pivot=d.pivot(index="Model",columns="Metric",values="Mean"); err=d.pivot(index="Model",columns="Metric",values="Std")
        fig,ax=plt.subplots(figsize=(9,5)); pivot.plot.bar(yerr=err,ax=ax,capsize=4); ax.set(ylabel="Test score",xlabel="",title="Main model comparison (two-seed mean ± SD)"); ax.tick_params(axis="x",rotation=0); savefig(fig,"fig5_model_comparison")
        cost=raw.groupby("Model").agg(Macro_F1_Mean=("macro_f1","mean"),Macro_F1_Std=("macro_f1","std"),Inference_ms_per_sample_Mean=("inference_ms_per_sample","mean"),Inference_ms_per_sample_Std=("inference_ms_per_sample","std"),Model_Size_MB=("model_size_mb","mean"),Parameters=("parameters","mean"),Device=("device","first")).reset_index()
        cost.to_csv(RESULTS/"figure_data"/"fig8_performance_vs_cost.csv",index=False)
        for x,name,xlabel in [("Inference_ms_per_sample_Mean","fig8_performance_vs_cost","Inference time (ms/sample)"),("Model_Size_MB","fig8b_model_size_vs_performance","Model size (MB)")]:
            fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(cost[x],cost.Macro_F1_Mean,s=90)
            for _,r in cost.iterrows(): ax.annotate(r.Model,(r[x],r.Macro_F1_Mean),xytext=(5,5),textcoords="offset points")
            ax.set(xlabel=xlabel,ylabel="Macro-F1",title="Performance vs computational cost"); savefig(fig,name)
        print("[OK] Figures 5, 8 and 8B")
    else: print("[UNAVAILABLE] Figures 5/8 require both seeds")
    # Bi-Encoder Recall@K
    br=[]
    for seed in CONFIG["seeds"]:
        m=read_metric("biencoder",seed)
        if m:
            for k in [1,3,5,10,20,50]: br.append({"Seed":seed,"K":k,"Recall":m["retrieval"].get(f"recall_at_{k}"),"Hit":m["retrieval"].get(f"hit_at_{k}")})
    if len(br)==12:
        bd=pd.DataFrame(br); bd.to_csv(RESULTS/"figure_data"/"fig5b_biencoder_retrieval.csv",index=False); bm=bd.groupby("K").agg(Recall=("Recall","mean"),Recall_SD=("Recall","std"),Hit=("Hit","mean"),Hit_SD=("Hit","std")).reset_index()
        fig,ax=plt.subplots(figsize=(8,5)); ax.errorbar(bm.K,bm.Recall,yerr=bm.Recall_SD,marker="o",label="Recall@K"); ax.errorbar(bm.K,bm.Hit,yerr=bm.Hit_SD,marker="s",label="Hit@K"); ax.set(xlabel="K",ylabel="Score",title="Bi-Encoder retrieval (two-seed mean ± SD)"); ax.legend(); savefig(fig,"fig5b_biencoder_retrieval")
        print("[OK] Figure 5B")
    # SecureBERT per-label Figures 6/6B/6C and errors.
    pls=[]
    for seed in CONFIG["seeds"]:
        p=RESULTS/"securebert_asl_augmented"/f"seed_{seed}"/"per_label_metrics.csv"
        if p.exists(): q=pd.read_csv(p); q["Seed"]=seed; pls.append(q)
    if len(pls)==2:
        a,b=pls; merged=a.merge(b,on="Technique_ID",suffixes=("_42","_123")); out=pd.DataFrame({"Technique_ID":merged.Technique_ID,"Train_Support":merged[["Train_Support_42","Train_Support_123"]].mean(1),"Test_Support":merged.Test_Support_42,"Seed42_F1":merged.F1_42,"Seed123_F1":merged.F1_123,"Mean_F1":merged[["F1_42","F1_123"]].mean(1),"Std_F1":merged[["F1_42","F1_123"]].std(1,ddof=1),"Frequency_Group":merged.Frequency_Group_42,"Mean_Precision":merged[["Precision_42","Precision_123"]].mean(1),"Mean_Recall":merged[["Recall_42","Recall_123"]].mean(1)})
        out=out.sort_values("Mean_F1",ascending=False); out.to_csv(RESULTS/"figure_data"/"fig6_per_label_f1.csv",index=False); out.to_csv(RESULTS/"tables"/"table6_per_label_results.csv",index=False)
        fig,ax=plt.subplots(figsize=(10,max(12,NUM_LABELS*.16))); ax.barh(out.Technique_ID[::-1],out.Mean_F1[::-1],color="#6A4C93"); ax.tick_params(axis="y",labelsize=5); ax.set(xlabel="Mean test F1",ylabel="Technique ID",title="SecureBERT 2.0 + ASL per-label F1"); savefig(fig,"fig6_per_label_f1")
        group=out.groupby("Frequency_Group").agg(Precision=("Mean_Precision","mean"),Recall=("Mean_Recall","mean"),F1=("Mean_F1","mean")).reindex(["Head","Medium","Tail"]); group.to_csv(RESULTS/"figure_data"/"fig6b_frequency_group_performance.csv")
        fig,ax=plt.subplots(figsize=(8,5)); group.plot.bar(ax=ax); ax.set(xlabel="Frequency group (training support)",ylabel="Mean per-label score",title="SecureBERT performance by label frequency"); ax.tick_params(axis="x",rotation=0); savefig(fig,"fig6b_frequency_group_performance")
        out[["Technique_ID","Train_Support","Mean_F1","Frequency_Group"]].to_csv(RESULTS/"figure_data"/"fig6c_support_vs_f1.csv",index=False)
        fig,ax=plt.subplots(figsize=(8,5)); colors={"Head":"#277DA1","Medium":"#F9C74F","Tail":"#F94144"}
        for group_name,group_df in out.groupby("Frequency_Group"): ax.scatter(group_df.Train_Support,group_df.Mean_F1,label=group_name,color=colors.get(group_name),alpha=.75)
        ax.set_xscale("log"); ax.legend(title="Frequency group"); ax.set(xlabel="Training label support (log scale)",ylabel="Mean test F1",title="Label support vs F1"); savefig(fig,"fig6c_support_vs_f1")
        bottom=out[out.Test_Support>0].nsmallest(10,"Mean_F1"); bottom.to_csv(RESULTS/"error_analysis"/"bottom10_f1_labels.csv",index=False)
        fig,ax=plt.subplots(figsize=(8,5)); ax.barh(bottom.Technique_ID,bottom.Mean_F1,color="#BC4749"); ax.set(xlabel="Mean F1",ylabel="Technique ID",title="Bottom 10 techniques by F1"); savefig(fig,"fig_error_bottom10_f1")
        fn_rows=[]
        for seed in CONFIG["seeds"]:
            z=np.load(RESULTS/"securebert_asl_augmented"/f"seed_{seed}"/"predictions.npz")
            yt=z["y_true"]; yp=z["per_label_predictions"]
            for j,tid in enumerate(classes):
                fn=int(((yt[:,j]==1)&(yp[:,j]==0)).sum()); support=int(yt[:,j].sum())
                fn_rows.append({"Seed":seed,"Technique_ID":tid,"FN":fn,"Support":support,"FN_rate":fn/support if support else np.nan,"Recall":recall_score(yt[:,j],yp[:,j],zero_division=0)})
        fn=pd.DataFrame(fn_rows).groupby("Technique_ID",as_index=False).agg(FN=("FN","mean"),Support=("Support","first"),FN_rate=("FN_rate","mean"),Recall=("Recall","mean")).nlargest(10,"FN")
        fn.to_csv(RESULTS/"error_analysis"/"top10_false_negative_labels.csv",index=False)
        fig,ax=plt.subplots(figsize=(8,5)); ax.barh(fn.Technique_ID[::-1],fn.FN[::-1],color="#D00000"); ax.set(xlabel="Mean false-negative count",ylabel="Technique ID",title="Top 10 techniques by false negatives"); savefig(fig,"fig_error_top10_false_negatives")
        print("[OK] Figures 6, 6B, 6C and error figures")
    else: print("[UNAVAILABLE] Figure 6 requires both SecureBERT seeds")
    # Bi-Encoder true-label rank distribution and retrieval by frequency group.
    rank_rows=[]; freq_rows=[]
    bins=[(1,1,"Rank 1"),(2,3,"Rank 2–3"),(4,5,"Rank 4–5"),(6,10,"Rank 6–10"),(11,20,"Rank 11–20"),(21,50,"Rank 21–50"),(51,10**9,">50")]
    for seed in CONFIG["seeds"]:
        p=RESULTS/"biencoder"/f"seed_{seed}"/"retrieval_predictions.npz"; lp=RESULTS/"biencoder"/f"seed_{seed}"/"per_label_metrics.csv"
        if not p.exists() or not lp.exists(): continue
        z=np.load(p); yt=z["y_true"]; ranks=z["true_label_ranks"]; ranking=z["rankings"]; true_ranks=ranks[yt.astype(bool)]
        for lo,hi,label in bins: rank_rows.append({"Seed":seed,"Rank_Bin":label,"True_Label_Count":int(((true_ranks>=lo)&(true_ranks<=hi)).sum())})
        groups=pd.read_csv(lp).set_index("Technique_ID").loc[classes,"Frequency_Group"].to_numpy()
        for group in ["Head","Medium","Tail"]:
            mask=groups==group; denom=yt[:,mask].sum()
            for k in [3,5,10]:
                retrieved=np.take_along_axis(yt,ranking[:,:k],axis=1)
                retrieved_ids=ranking[:,:k]; hits=sum(yt[i,j] for i in range(len(yt)) for j in retrieved_ids[i] if mask[j])
                freq_rows.append({"Seed":seed,"Frequency_Group":group,"K":k,"Recall":float(hits/denom) if denom else np.nan})
    if len(rank_rows)==14:
        rd=pd.DataFrame(rank_rows); rd.to_csv(RESULTS/"figure_data"/"fig_biencoder_true_label_rank_distribution.csv",index=False); order_bins=[x[2] for x in bins]; rm=rd.groupby("Rank_Bin").True_Label_Count.mean().reindex(order_bins)
        fig,ax=plt.subplots(figsize=(9,5)); ax.bar(rm.index,rm.values,color="#277DA1"); ax.tick_params(axis="x",rotation=25); ax.set(xlabel="Ground-truth label rank",ylabel="Mean true-label count",title="Bi-Encoder true-label rank distribution"); savefig(fig,"fig_biencoder_true_label_rank_distribution")
        fd=pd.DataFrame(freq_rows); fd.to_csv(RESULTS/"figure_data"/"fig_biencoder_recall_by_frequency.csv",index=False); fm=fd.groupby(["Frequency_Group","K"]).Recall.mean().unstack()
        fig,ax=plt.subplots(figsize=(8,5)); fm.reindex(["Head","Medium","Tail"]).plot.bar(ax=ax); ax.set(xlabel="Training-frequency group",ylabel="Recall",title="Bi-Encoder Recall@K by label frequency"); ax.tick_params(axis="x",rotation=0); savefig(fig,"fig_biencoder_recall_by_frequency")
        print("[OK] Bi-Encoder rank and frequency figures")
    # Training curves for every completed run.
    for model_dir,prefix in [("securebert_asl","securebert"),("biencoder","biencoder")]:
        for seed in CONFIG["seeds"]:
            p=RESULTS/model_dir/f"seed_{seed}"/"training_history.csv"
            if not p.exists(): continue
            h=pd.read_csv(p); fig,ax=plt.subplots(figsize=(6,4)); ax.plot(h.epoch,h.train_loss,marker="o"); ax.set(xlabel="Epoch",ylabel="Train loss",title=f"{prefix} seed {seed} training loss"); savefig(fig,f"training/{prefix}_seed{seed}_loss")
            metric="val_macro_f1" if model_dir=="securebert_asl_augmented" else "val_recall_at_5"; fig,ax=plt.subplots(figsize=(6,4)); ax.plot(h.epoch,h[metric],marker="o"); ax.set(xlabel="Epoch",ylabel=metric.replace("_"," "),title=f"{prefix} seed {seed} validation metric"); savefig(fig,f"training/{prefix}_seed{seed}_metrics")

def generate_ablation_figure():
    rows=[]
    for seed in CONFIG["seeds"]:
        m=read_metric("securebert_asl",seed)
        if m:
            rows.extend([{"Configuration":"Global threshold","Seed":seed,"Macro_F1":m["test_global"]["macro_f1"]},{"Configuration":"Per-label threshold","Seed":seed,"Macro_F1":m["test_per_label"]["macro_f1"]}])
    if len(rows)==4:
        d=pd.DataFrame(rows); wide=d.pivot(index="Configuration",columns="Seed",values="Macro_F1").reset_index(); wide.columns=["Configuration","Seed42_Macro_F1","Seed123_Macro_F1"]; wide["Mean_Macro_F1"]=wide.iloc[:,1:3].mean(1); wide["Std_Macro_F1"]=wide.iloc[:,1:3].std(1,ddof=1); base=wide.loc[wide.Configuration=="Global threshold","Mean_Macro_F1"].iloc[0]; wide["Delta_vs_Baseline"]=wide.Mean_Macro_F1-base; wide.to_csv(RESULTS/"figure_data"/"fig7_ablation_improvement.csv",index=False)
        fig,ax=plt.subplots(figsize=(7,4)); ax.bar(wide.Configuration,wide.Mean_Macro_F1,yerr=wide.Std_Macro_F1,capsize=4,color=["#577590","#F9844A"]); ax.set(ylabel="Macro-F1",title="Thresholding ablation (two-seed mean ± SD)"); savefig(fig,"fig7_ablation_improvement"); print("[PARTIAL] Figure 7: available threshold ablation only")
    else: print("[UNAVAILABLE] Figure 7 requires both SecureBERT seeds")

def validate_outputs():
    checks={f"SecureBERT seed {s}":RESULTS/"securebert_asl_augmented"/f"seed_{s}"/"metrics.json" for s in CONFIG["seeds"]}
    checks.update({f"Bi-Encoder seed {s}":RESULTS/"biencoder"/f"seed_{s}"/"metrics.json" for s in CONFIG["seeds"]})
    for n in [1,2,3,5,6,7,8]:
        stem={1:"fig1_label_distribution",2:"fig2_labels_per_sample",3:"fig3_label_cooccurrence",5:"fig5_model_comparison",6:"fig6_per_label_f1",7:"fig7_ablation_improvement",8:"fig8_performance_vs_cost"}[n]
        checks[f"Figure {n}"]=RESULTS/"figures"/f"{stem}.png"
    print("\n"+"="*58+"\nFINAL KAGGLE PIPELINE STATUS\n"+"="*58)
    for name,path in checks.items(): print(f"{name:32} {'OK' if path.exists() else 'FAILED'}  {path}")
    print("Results directory:",RESULTS)

## 7. Run the complete pipeline

This is the only execution cell needed. Completed runs are skipped unless `CONFIG["force_rerun"] = True`. Figure generation runs only after all four metrics files exist.

In [ ]:
if RUN_ALL:
    for seed in CONFIG["seeds"]:
        run_securebert(seed)
    # BI-ENCODER DISABLED
